# Ill-Conditioned Optimization Practice

**Topic:** Optimization &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(1000, 30)`, `y_train`: linear ground truth, tiny noise, but feature scales spanning ~4 orders of magnitude.
- `X_test` `(400, 30)`: predict these. `y_test` is used by the evaluation cell.
- Try vanilla GD on the raw features first and watch it crawl.

## Your task

Fit the linear model **iteratively** (no `np.linalg.solve/lstsq`, no sklearn estimators):

1. Either standardize the features (remember to un-scale your weights or scale the test set identically), or implement momentum/Adam updates on the raw problem.
2. Keep it under ~20,000 iterations.
3. Set `predictions` to your outputs on `X_test`.

## Evaluation

Test R² must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(241)
d = 30
scales = np.logspace(-2, 2, d)          # feature scales span 4 orders of magnitude
w_true = rng.normal(0, 2.0, d)

def make(n, seed):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, d)) * scales
    y = X @ (w_true / scales) + r.normal(0, 0.5, n)
    return X, y

X_train, y_train = make(1000, 251)
X_test, y_test = make(400, 252)
print(f"feature scale ratio: {scales.max() / scales.min():.0f}x")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_R2 = 0.9

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions, dtype=float).ravel()
_true = np.asarray(y_test, dtype=float).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

_rmse = float(np.sqrt(np.mean((_pred - _true) ** 2)))
_ss_res = float(np.sum((_true - _pred) ** 2))
_ss_tot = float(np.sum((_true - _true.mean()) ** 2))
r2 = 1.0 - _ss_res / _ss_tot
print("=" * 52)
print(f"Test RMSE : {_rmse:.4f}")
print(f"Test R^2  : {r2:.4f}   (target: >= {TARGET_R2:.2f})")
print(f"Worst absolute error: {float(np.max(np.abs(_pred - _true))):.4f}")
print("=" * 52)
if r2 >= TARGET_R2:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

Standardize the features so every direction has comparable curvature, run plain gradient descent in the scaled space, and apply the identical transform to the test matrix at prediction time.

```python
import numpy as np

mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
Ztr = (X_train - mu) / sd
Zb = np.column_stack([Ztr, np.ones(len(Ztr))])

w = np.zeros(Zb.shape[1])
lr = 0.4
for _ in range(10000):
    grad = 2.0 * Zb.T @ (Zb @ w - y_train) / len(y_train)
    w -= lr * grad

Zte = np.column_stack([(X_test - mu) / sd, np.ones(len(X_test))])
predictions = Zte @ w
```
